# pinn_ndof_chain_sim.ipynb — n-DOF Impact Damper PINN  (one impactor per DOF)

Calls `pinn_ndof_chain.py` for the generalised `PIPNNs` class and helpers.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.font_manager import FontProperties
import os

from pinn_ndof_chain import PIPNNs, find_impact_times, impact_velocity_update, propagate_ics, newmark_beta

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
np.random.seed(1234)
tf.compat.v1.set_random_seed(1234)
print("TF version:", tf.__version__)

## System parameters

In [ ]:
# ── Number of DOFs ─────────────────────────────────────────────────────────
n_dof = 10          # each DOF has its own internal impactor

# ── Physical parameters ────────────────────────────────────────────────────
m_x = 1.0           # primary mass (uniform)
m_y = 0.1           # impactor mass (uniform)
k   = 100.0         # neighbour spring stiffness
c   = 0.5           # damping coefficient (proportional to K)
D   = 1.0           # impact gap (same for all DOFs)
r   = 0.7           # coefficient of restitution

phi1 = np.array([[1.0]])    # forcing amplitude
phi2 = np.array([[1.0]])    # forcing frequency  F = phi1*sin(phi2*pi*(t+phi))

# ── Mass matrix ────────────────────────────────────────────────────────────
M_total = np.diag(m_x * np.ones(n_dof))

# ── Stiffness matrix (tridiagonal chain) ──────────────────────────────────
K_total = np.zeros((n_dof, n_dof))
for i in range(n_dof):
    K_total[i, i] = 2*k if 0 < i < n_dof-1 else k
    if i > 0:         K_total[i, i-1] = -k
    if i < n_dof-1:   K_total[i, i+1] = -k

# ── Damping matrix (proportional to K) ────────────────────────────────────
C_total = (c / k) * K_total

print("M ="); print(np.diag(M_total))
print("K (diag) =", np.diag(K_total))
print("C (diag) =", np.diag(C_total))

## Impact velocity-update matrix  $A^{-1}B$

All DOFs share the same primary and impactor mass, so one matrix suffices.

In [ ]:
# Momentum + restitution:
#   A [xt+; yt+] = B [xt-; yt-]
A = np.array([[m_x, m_y],
              [1.0, -1.0]])
B = np.array([[m_x, m_y],
              [-r,   r  ]])
A_inv_B = np.linalg.inv(A) @ B
print("A_inv_B ="); print(A_inv_B)

## Newmark-beta reference (undamped, no impactors)

Full 20 s run to check free structural response.

In [ ]:
T_ref  = 20.0
dt_ref = 0.001
num_ref = int(T_ref / dt_ref) + 1
t_ref   = np.linspace(0, T_ref, num_ref).reshape(-1, 1)

F_ref = np.zeros((n_dof, num_ref))
F_ref[n_dof-1, :] = float(phi1) * np.sin(float(phi2)*np.pi*t_ref).flatten()

u_NM_ref, ut_NM_ref, _ = newmark_beta(
    M_total, C_total, K_total, F_ref, dt_ref, num_ref, n_dof)

plt.figure(figsize=(10, 3))
for i in range(n_dof):
    plt.plot(t_ref, u_NM_ref[i, :], lw=0.8, label=f'DOF {i+1}')
plt.xlim(0, T_ref); plt.xlabel('Time (s)'); plt.ylabel('Displacement')
plt.title('Free response — all DOFs (no impactors, Newmark-β)')
plt.legend(ncol=5, fontsize=7)
plt.tight_layout(); plt.show()

## PINN hyper-parameters

In [ ]:
num_neurons = 64
layers = [1, num_neurons, num_neurons, n_dof]   # deeper net for larger system

hyp_ini_weight_loss = np.array([100.0, 1.0])    # [beta_icx, beta_fx]
optimizer_LB_value  = True

lb = np.array([0.0])
ub = np.array([6.0])   # upper bound covers the longest allowed segment

## Initial conditions  (segment 1)

In [ ]:
t0          = np.array([[0.0]])
T01         = np.array([4.0])
num01       = 1000
nIter01     = 1000

x0_total    = np.zeros((1, n_dof))   # all primary DOFs at rest
xt0_total   = np.zeros((1, n_dof))

y0_01       = np.zeros(n_dof)        # impactors start flush with primary DOF
yt0_01      = np.zeros(n_dof)        # impactors at rest

phi01       = np.array([[0.0]])      # accumulated phase offset at start

t01 = np.linspace(0, float(T01[0]), num01).reshape(-1, 1)
print("Segment 1 time window:", float(T01[0]), "s")

## Train segment 1

In [ ]:
print("\nSimulating Impact 1 ...")
model01 = PIPNNs(
    lb, ub,
    t0, t01,
    x0_total, xt0_total,
    y0_01, yt0_01,
    M_total, K_total, D, n_dof,
    phi01, phi1, phi2,
    layers,
    hyp_ini_weight_loss,
    C=C_total,
    optimizer_LB=optimizer_LB_value,
)
model01.train(nIter=nIter01, optimizer_LB=optimizer_LB_value)

## Phase 2 — find impact times (segment 1)

In [ ]:
t_impacts01, hit01 = find_impact_times(
    model01, y0_01, yt0_01, D, float(T01[0]))

first_dof01   = int(np.argmin(t_impacts01))
t_impact01    = t_impacts01[first_dof01]

print(f"\nFirst impact: DOF {first_dof01+1} at t = {t_impact01:.5f} s")
print("All impact times:", np.round(t_impacts01, 4))

## Trajectory simulation and IC propagation (segment 1 → 2)

In [ ]:
# Predict at segment boundary
t_sim01 = np.linspace(0, t_impact01, num01+1).reshape(-1, 1)
x_sim01, xt_sim01, xtt_sim01 = model01.predict(t_sim01)

# Free-flight impactor trajectories
y_sim01  = (y0_01 + yt0_01 * t_sim01).T    # (n_dof, num01+1)
yt_sim01 = np.tile(yt0_01[:, None], (1, num01+1))

# Newmark reference for this segment
F01 = np.zeros((n_dof, num01+1))
F01[n_dof-1, :] = float(phi1) * np.sin(float(phi2)*np.pi*t_sim01).flatten()
x_NM01, xt_NM01, _ = newmark_beta(
    M_total, C_total, K_total, F01,
    t_impact01/num01, num01+1, n_dof,
    x0_total[0], xt0_total[0])

# Predict at impact
x1_01, xt1_01, _ = model01.predict(np.array([[t_impact01]]))

# Propagate ICs
x0_02, xt0_02, y0_02, yt0_02 = propagate_ics(
    model01, t_impact01,
    x1_01, xt1_01,
    y0_01, yt0_01,
    first_dof01,
    m_x * np.ones(n_dof), m_y * np.ones(n_dof), r,
    A_inv_B,
)
phi02 = phi01 + t_impact01
print("ICs for segment 2 ready.")

## Quick plot — segment 1

In [ ]:
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size']   = 12

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax = axes[0]
for i in range(n_dof):
    ax.plot(t_sim01, x_NM01[i, :],  lw=0.8, color=f'C{i}', linestyle='-')
    ax.plot(t_sim01, x_sim01[:, i], lw=1.2, color=f'C{i}', linestyle='--')
ax.axvline(t_impact01, color='k', lw=1, linestyle=':')
ax.set_ylabel('Displacement')
ax.set_title('Segment 1 — all DOFs  (solid: Newmark, dashed: PINN)')

ax = axes[1]
for i in range(n_dof):
    rel = x_sim01[:, i] - y_sim01[i, :]
    ax.plot(t_sim01, rel, lw=1.0, color=f'C{i}', label=f'DOF {i+1}')
ax.axhline( D, color='r', lw=1, linestyle='--', label='±D')
ax.axhline(-D, color='r', lw=1, linestyle='--')
ax.axvline(t_impact01, color='k', lw=1, linestyle=':')
ax.set_xlabel('Time (s)'); ax.set_ylabel('x_i − y_i')
ax.set_title('Relative displacement (primary − impactor)')
ax.legend(ncol=5, fontsize=8)

plt.tight_layout(); plt.show()

## Multi-segment loop

Run `n_segments` impact events end-to-end.

In [ ]:
n_segments  = 10   # total impacts to simulate
T_seg       = np.array([4.0])
num_seg     = 1000
nIter_seg   = 1000

# Storage for stitching
all_t_sim   = []    # list of shifted time arrays (n_dof+1, num_seg+1)
all_x_sim   = []    # PINN primary displacements (num_seg+1, n_dof)
all_xt_sim  = []    # PINN primary velocities
all_x_NM    = []    # Newmark primary displacements (n_dof, num_seg+1)
all_y_sim   = []    # impactor displacements (n_dof, num_seg+1)
all_t_imp   = []    # impact times per segment (n_dof,)

# Carry current ICs forward
cur_x0   = x0_02.copy()
cur_xt0  = xt0_02.copy()
cur_y0   = y0_02.copy()
cur_yt0  = yt0_02.copy()
cur_phi  = phi02.copy()
t_cumulative = float(phi02)

# Also store segment 1 results (already computed above)
all_t_sim.append(t_sim01.flatten() + 0.0)
all_x_sim.append(x_sim01)
all_xt_sim.append(xt_sim01)
all_x_NM.append(x_NM01)
all_y_sim.append(y_sim01.T)
all_t_imp.append(t_impacts01)

for seg in range(2, n_segments + 1):
    print(f"\n{'='*55}")
    print(f"  Simulating Impact {seg} ...")
    print(f"{'='*55}")

    t_seg_arr = np.linspace(0, float(T_seg[0]), num_seg).reshape(-1, 1)

    model = PIPNNs(
        lb, ub,
        t0, t_seg_arr,
        cur_x0, cur_xt0,
        cur_y0, cur_yt0,
        M_total, K_total, D, n_dof,
        cur_phi, phi1, phi2,
        layers, hyp_ini_weight_loss,
        C=C_total,
        optimizer_LB=optimizer_LB_value,
    )
    model.train(nIter=nIter_seg, optimizer_LB=optimizer_LB_value)

    t_impacts, hit = find_impact_times(model, cur_y0, cur_yt0, D, float(T_seg[0]))
    first_dof = int(np.argmin(t_impacts))
    t_imp     = t_impacts[first_dof]
    print(f"  → DOF {first_dof+1} impacts first at t = {t_imp:.5f} s")

    t_sim  = np.linspace(0, t_imp, num_seg+1).reshape(-1, 1)
    x_sim, xt_sim, _ = model.predict(t_sim)

    y_sim_seg = (cur_y0 + cur_yt0 * t_sim).T    # (n_dof, num_seg+1)

    F_s = np.zeros((n_dof, num_seg+1))
    F_s[n_dof-1, :] = float(phi1) * np.sin(
        float(phi2)*np.pi*(t_sim + float(cur_phi))).flatten()
    x_NM_seg, xt_NM_seg, _ = newmark_beta(
        M_total, C_total, K_total, F_s,
        t_imp/num_seg, num_seg+1, n_dof,
        cur_x0[0], cur_xt0[0])

    all_t_sim.append(t_sim.flatten() + t_cumulative)
    all_x_sim.append(x_sim)
    all_xt_sim.append(xt_sim)
    all_x_NM.append(x_NM_seg)
    all_y_sim.append(y_sim_seg.T)
    all_t_imp.append(t_impacts)

    # Propagate ICs
    x1_s, xt1_s, _ = model.predict(np.array([[t_imp]]))
    next_x0, next_xt0, next_y0, next_yt0 = propagate_ics(
        model, t_imp,
        x1_s, xt1_s,
        cur_y0, cur_yt0,
        first_dof,
        m_x * np.ones(n_dof), m_y * np.ones(n_dof), r,
        A_inv_B,
    )
    t_cumulative += t_imp
    cur_x0   = next_x0
    cur_xt0  = next_xt0
    cur_y0   = next_y0
    cur_yt0  = next_yt0
    cur_phi  = cur_phi + t_imp

print("\nAll segments complete.")

## Stitch all segments into global time series

In [ ]:
t_total      = np.concatenate(all_t_sim)
x_PINN_total = np.vstack(all_x_sim)           # (N_total, n_dof)
xt_PINN_total= np.vstack(all_xt_sim)
x_NM_total   = np.hstack(all_x_NM)            # (n_dof, N_total)
y_total      = np.vstack(all_y_sim)           # (N_total, n_dof)

# Accumulated impact times for vertical marker lines
t_impacts_cumulative = []
cumul = 0.0
for t_arr in all_t_sim:
    cumul = t_arr[-1]
    t_impacts_cumulative.append(cumul)

print(f"Total time points: {len(t_total)}")
print(f"Global time span:  0 → {t_total[-1]:.3f} s")

## Plot displacements — all DOFs

In [ ]:
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size']   = 13

fig, axes = plt.subplots(n_dof, 1, figsize=(12, 2.5*n_dof), sharex=True)

for i, ax in enumerate(axes):
    ax.plot(t_total, x_NM_total[i, :],    lw=0.9, color='C0', label='Newmark')
    ax.plot(t_total, x_PINN_total[:, i],  lw=1.3, color='C1', linestyle='--', label='PINN')
    for t_v in t_impacts_cumulative:
        ax.axvline(t_v, color='gray', lw=0.6, linestyle=':')
    ax.set_ylabel(rf'$x_{i+1}$')
    ax.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Displacement response — {n_dof}-DOF chain with impact dampers', fontsize=14)
plt.tight_layout()
plt.savefig('displacement_all_dofs.png', dpi=150)
plt.show()

## Plot relative displacements  $x_i - y_i$

In [ ]:
fig, axes = plt.subplots(n_dof, 1, figsize=(12, 2.5*n_dof), sharex=True)

for i, ax in enumerate(axes):
    rel = x_PINN_total[:, i] - y_total[:, i]
    ax.plot(t_total, rel, lw=1.2, color=f'C{i % 10}')
    ax.axhline( D, color='r', lw=1.0, linestyle='--', label='±D')
    ax.axhline(-D, color='r', lw=1.0, linestyle='--')
    for t_v in t_impacts_cumulative:
        ax.axvline(t_v, color='gray', lw=0.6, linestyle=':')
    ax.set_ylabel(rf'$x_{i+1}-y_{i+1}$')

axes[-1].set_xlabel('Time (s)')
axes[0].legend(fontsize=9)
fig.suptitle('Relative displacement (primary − impactor) per DOF', fontsize=14)
plt.tight_layout()
plt.savefig('relative_disp.png', dpi=150)
plt.show()

## Impact times summary — bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

x_pos = np.arange(n_segments)
width = 0.06
colors = [f'C{i}' for i in range(n_dof)]

for i in range(n_dof):
    times_dof_i = [all_t_imp[s][i] for s in range(n_segments)]
    offset = (i - n_dof/2 + 0.5) * width
    ax.bar(x_pos + offset, times_dof_i, width,
           label=f'DOF {i+1}', color=colors[i])

ax.set_xticks(x_pos)
ax.set_xticklabels([f'Seg {s+1}' for s in range(n_segments)])
ax.set_xlabel('Segment'); ax.set_ylabel('Impact time within segment (s)')
ax.set_title(f'Phase 2 root-found impact times — {n_dof} DOFs × {n_segments} segments')
ax.legend(ncol=5, fontsize=9)
plt.tight_layout()
plt.savefig('impact_times.png', dpi=150)
plt.show()

# Table
print(f"{'Seg':>4}  {'1st-hit DOF':>11}  {'t* (s)':>8}")
for s in range(n_segments):
    fdof = int(np.argmin(all_t_imp[s])) + 1
    ti   = float(np.min(all_t_imp[s]))
    print(f"{s+1:>4}  {'DOF '+str(fdof):>11}  {ti:>8.4f}")

## Training loss convergence (segment 1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(model01.loss_log,     label='Total',  lw=1.5)
ax.semilogy(model01.loss_icx_log, label='IC',     lw=1.0, linestyle='--')
ax.semilogy(model01.loss_fx_log,  label='ODE',    lw=1.0, linestyle=':')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('Segment 1 — training loss')
ax.legend(); plt.tight_layout(); plt.show()